In [1]:
import ee
import pandas as pd
import json
from glob import glob
import geemap
import pyproj
from geopandas import geopandas as gpd
from shapely.geometry import Point
from copy import deepcopy
import numpy as np
import os
import ast
from google.cloud import storage

In [ ]:
# from google.colab import drive
# drive.flush_and_unmount()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
ee.Authenticate()
ee_project = "corestack1-dev-alpha"
ee.Initialize(project="core-stack-dev-2")#"corestack1-dev-alpha")

In [4]:
client = storage.Client()
bucket = client.get_bucket('core_stack')

In [5]:
best_month_dict = {'Eastern Plateau & Hills Region': 'cc_12',
                   'Middle Gangetic Plain Region': 'cc_10',
                   'Lower Gangetic Plain Region': 'cc_9',
                   'Western Himalayan Region': 'cc_8',
                   'Eastern Himalayan Region': 'cc_10',
                   'Upper Gangetic Plain Region': 'cc_9',
                   'Trans Gangetic Plain Region': 'cc_9',
                   'Central Plateau & Hills Region': 'cc_7',
                   'Western Plateau and Hills Region': 'cc_11',
                   'Southern Plateau and Hills Region': 'cc_8',
                   'East Coast Plains & Hills Region': 'cc_12'}

In [6]:
# agroclimatic_zone = 'Eastern Plateau & Hills Region'

# agroclimatic_zone = 'Lower Gangetic Plain Region'

agroclimatic_zone = 'Middle Gangetic Plain Region'

# agroclimatic_zone = 'Western Himalayan Region'

# agroclimatic_zone = 'Eastern Himalayan Region'

# agroclimatic_zone = 'Upper Gangetic Plain Region'

# agroclimatic_zone = 'Trans Gangetic Plain Region'

# agroclimatic_zone = 'Central Plateau & Hills Region'

# agroclimatic_zone = 'Western Plateau and Hills Region'

# agroclimatic_zone = 'Southern Plateau and Hills Region'

# agroclimatic_zone = 'East Coast Plains & Hills Region'

In [7]:
agroclimaticZone_acronym_dict = {'Eastern Plateau & Hills Region': 'EPAHR',
                               'Southern Plateau and Hills Region': 'SPAHR',
                               'East Coast Plains & Hills Region': 'ECPHR',
                               'Western Plateau and Hills Region': 'WPAHR',
                               'Central Plateau & Hills Region': 'CPAHR',
                               'Lower Gangetic Plain Region': 'LGPR',
                                'Middle Gangetic Plain Region': 'MGPR',
                                'Eastern Himalayan Region': 'EHR',
                                'Western Himalayan Region': 'WHR',
                                'Upper Gangetic Plain Region': 'UGPR',
                                'Trans Gangetic Plain Region': 'TGPR',
                                'West Coast Plains & Ghat Region': 'WCPGR',
                                'Gujarat Plains & Hills Region': 'GPHR',
                                'Western Dry Region': 'WDR'}

In [ ]:
# acz_list = ['Western Himalayan Region', 'Eastern Himalayan Region', 'Lower Gangetic Plain Region',
#             'Middle Gangetic Plain Region', 'Upper Gangetic Plain Region', 'Trans Gangetic Plain Region',
#             'Eastern Plateau & Hills Region', 'Central Plateau & Hills Region', 'Western Plateau and Hills Region',
#             'Southern Plateau and Hills Region', 'East Coast Plains & Hills Region']

# acz_list = ['Eastern Plateau & Hills Region']

In [8]:
def upload_file_to_gcs(local_file_path, file_name):
    blob = bucket.blob(f'nrm_tree_health/compiled_results/{file_name}.csv')        # GCS path
    blob.upload_from_filename(local_file_path)

    print("Upload complete.")

def export_to_gee(file_name):
  # CSV GCS path
  gcs_path = f'gs://core_stack/nrm_tree_health/compiled_results/{file_name}.csv'

  # Create task ID and run table ingestion
  task_id = ee.data.newTaskId()[0]
  asset_id = f'projects/{ee_project}/assets/tree_characteristics/{file_name}'

  manifest = {
    'id': asset_id,
    'sources': [
        {
            'primaryPath': gcs_path,
            'additionalPaths': []
        }
    ]
  }

  ee.data.startTableIngestion(task_id, manifest)
  print("Ingestion task started:", task_id)

In [9]:
# Set the years for which results need to be combined
years = ['2016','2017', '2018','2019','2020','2021','2022','2023','2024']

# Combine CSVs CCD

In [ ]:
df = pd.read_csv(f'drive/MyDrive/TreeHealth/Agroclimatic_regions/{agroclimatic_zone}.csv')
dist_list = list(df['Name'])
print(len(dist_list))
print(dist_list)

119
['East Godavari', 'Srikakulam', 'Visakhapatnam', 'Vizianagaram', 'AurangabadB', 'Banka', 'Gaya', 'Jamui', 'Nawada', 'Rohtas', 'Baloda Bazar', 'Balod', 'BalrampurC', 'Bastar', 'Bemetara', 'BijapurC', 'BilaspurC', 'Dantewada', 'Dhamtari', 'Durg', 'Gariaband', 'Janjgir-Champa', 'Jashpur', 'Kabeerdham', 'Kondagaon', 'Korba', 'Koriya', 'Mahasamund', 'Mungeli', 'Narayanpur', 'Raigarh', 'Raipur', 'Rajnandgaon', 'Sukma', 'Surajpur', 'Surguja', 'Uttar Bastar Kanker', 'Bokaro', 'Chatra', 'Deoghar', 'Dhanbad', 'Dumka', 'Garhwa', 'Giridih', 'Godda', 'Gumla', 'Hazaribagh', 'Jamtara', 'Khunti', 'Kodarma', 'Latehar', 'Lohardaga', 'Pakur', 'Palamu', 'Pashchimi Singhbhum', 'Purbi Singhbhum', 'Ramgarh', 'Ranchi', 'Sahibganj', 'Saraikela-kharsawan', 'Simdega', 'Anuppur', 'Balaghat', 'Dindori', 'Jabalpur', 'Katni', 'Mandla', 'Rewa', 'Satna', 'Seoni', 'Shahdol', 'Sidhi', 'Singrauli', 'Umaria', 'Bhandara', 'Chandrapur', 'Garhchiroli', 'Gondiya', 'Nagpur', 'Wardha', 'Yavatmal', 'Anugul', 'Balangir', 'Bal

In [ ]:
best_month = best_month_dict[agroclimatic_zone]
chunk_size=500_000
max_rows=80_000_000

for year in years:
    print(year)
    result_num = 0
    dst_num = 0
    row_counter = 0

    path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results/{year}/'
    os.makedirs(path, exist_ok=True)
    drive_path = f"{path}/result_{result_num}.csv"
    first_write = True  # controls header

    for district in dist_list:
        print(dst_num, district)
        dst_num += 1
        try:
            reader = pd.read_csv(
                f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_monthly_cc.csv',
                chunksize=chunk_size
            )
        except:
            continue

        for df_chunk in reader:
            if df_chunk.shape[0] > 0:
              # print(df_chunk)
              # Add "cc" column from best month
              df_chunk['cc'] = df_chunk[best_month]

              # Append to current output file
              df_chunk.to_csv(drive_path, mode='a',
                              header=first_write, index=False)
              first_write = False

              row_counter += len(df_chunk)

              # Split file if max_rows exceeded
              if row_counter >= max_rows:
                  print(f'Saving result_{result_num}.csv with {row_counter:,} rows')

                  file_name = f"ccd_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
                  upload_file_to_gcs(drive_path, file_name)
                  export_to_gee(file_name)

                  # reset counters
                  result_num += 1
                  row_counter = 0
                  drive_path = f"{path}/result_{result_num}.csv"
                  first_write = True  # new file needs header

    # flush leftovers
    if row_counter > 0:
        print(f'Saving result_{result_num}.csv with {row_counter:,} rows')
        file_name = f"ccd_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
        upload_file_to_gcs(drive_path, file_name)
        export_to_gee(file_name)


2023
0 East Godavari
1 Srikakulam
2 Visakhapatnam
3 Vizianagaram
4 AurangabadB
5 Banka
6 Gaya
7 Jamui
8 Nawada
9 Rohtas
10 Baloda Bazar
11 Balod
12 BalrampurC
13 Bastar
14 Bemetara
15 BijapurC
16 BilaspurC
17 Dantewada
18 Dhamtari
19 Durg
20 Gariaband
21 Janjgir-Champa
22 Jashpur
23 Kabeerdham
24 Kondagaon
25 Korba
26 Koriya
Saving result_0.csv with 80,497,091 rows
Upload complete.
Ingestion task started: c73487a3-3340-4ce0-afe7-141323f3c52a
27 Mahasamund
28 Mungeli
29 Narayanpur
30 Raigarh
31 Raipur
32 Rajnandgaon
33 Sukma
34 Surajpur
35 Surguja
36 Uttar Bastar Kanker
37 Bokaro
38 Chatra
39 Deoghar
40 Dhanbad
41 Dumka
42 Garhwa
43 Giridih
Saving result_1.csv with 80,298,124 rows


Upload complete.


Ingestion task started: 1469ba67-3aff-4e72-997c-39bbba391b60
44 Godda
45 Gumla
46 Hazaribagh
47 Jamtara
48 Khunti
49 Kodarma
50 Latehar
51 Lohardaga
52 Pakur
53 Palamu
54 Pashchimi Singhbhum
55 Purbi Singhbhum
56 Ramgarh
57 Ranchi
58 Sahibganj
59 Saraikela-kharsawan
60 Simdega
61 Anuppur
62 Balaghat
63 Dindori
64 Jabalpur
65 Katni
66 Mandla
67 Rewa
68 Satna
69 Seoni
70 Shahdol
71 Sidhi
Saving result_2.csv with 80,399,602 rows


Upload complete.


Ingestion task started: 17b547b5-7742-4840-806d-2fb7800bc9ce
72 Singrauli
73 Umaria
74 Bhandara
75 Chandrapur
76 Garhchiroli
77 Gondiya
78 Nagpur
79 Wardha
80 Yavatmal
81 Anugul
82 Balangir
83 Baleshwar
84 Bargarh
85 Bauda
86 Bhadrak
87 Cuttack
88 Debagarh
89 Dhenkanal
Saving result_3.csv with 80,196,991 rows


...

Upload complete.


Ingestion task started: dafeda5b-16c0-423d-a117-240a643c4431
27 Mahasamund
28 Mungeli
29 Narayanpur
30 Raigarh
31 Raipur
32 Rajnandgaon
33 Sukma
34 Surajpur
35 Surguja
36 Uttar Bastar Kanker
37 Bokaro
38 Chatra
39 Deoghar
40 Dhanbad
41 Dumka
42 Garhwa
43 Giridih
Saving result_1.csv with 80,493,525 rows


Upload complete.


Ingestion task started: 248f4208-704a-4ba9-85d1-9bead9490c53
44 Godda
45 Gumla
46 Hazaribagh
47 Jamtara
48 Khunti
49 Kodarma
50 Latehar
51 Lohardaga
52 Pakur
53 Palamu
54 Pashchimi Singhbhum
55 Purbi Singhbhum
56 Ramgarh
57 Ranchi
58 Sahibganj
59 Saraikela-kharsawan
60 Simdega
61 Anuppur
62 Balaghat
63 Dindori
64 Jabalpur
65 Katni
66 Mandla
67 Rewa
68 Satna
69 Seoni
70 Shahdol
71 Sidhi
Saving result_2.csv with 80,264,095 rows
Upload complete.
Ingestion task started: 3cde4a32-1589-462e-b073-f84f97e6cd27


OSError: [Errno 5] Input/output error

In [ ]:
# best_month = best_month_dict[agroclimatic_zone]
# for year in years:
#   print(year)
#   df_con = pd.DataFrame()
#   df_len = 0
#   result_num = 0
#   dst_num = 0
#   for district in dist_list:
#     print(dst_num, district)
#     dst_num += 1
#     try:
#       df = pd.read_csv(f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_monthly_cc.csv')
#     except:
#         continue
#     df_len += len(df)
#     df_con = pd.concat([df_con, df])
#     del(df)
#     print(df_len)


#     if df_len > 80000000:
#       print(f'Saving result_{result_num}.csv')
#       df_con['cc'] = df_con[best_month]
#       path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results/{year}/'
#       if not os.path.exists(path):
#         os.makedirs(path)
#       drive_path = f'{path}/result_{result_num}.csv'
#       df_con.to_csv(f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results/{year}/result_{result_num}.csv', index=False)
#       file_name = f"ccd_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
#       upload_file_to_gcs(drive_path, file_name)
#       export_to_gee(file_name)

#       result_num += 1
#       del(df_con)
#       df_len = 0
#       df_con = pd.DataFrame()

#   if (len(df_con) > 0):
#     print(f'Saving result_{result_num}.csv')
#     df_con['cc'] = df_con[best_month]
#     path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results/{year}/'
#     if not os.path.exists(path):
#         os.makedirs(path)
#     drive_path = f'{path}/result_{result_num}.csv'
#     df_con.to_csv(drive_path, index=False)

#     file_name = f"ccd_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
#     upload_file_to_gcs(drive_path, file_name)
#     export_to_gee(file_name)

#     result_num += 1
#     del(df_con)
#     df_len = 0
#     df_con = pd.DataFrame()

In [ ]:
# year = '2016'
# path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results/{year}/'
# for result_num in range(1):
#   drive_path = f"{path}/result_{result_num}.csv"
#   reader = pd.read_csv(drive_path, chunksize=100)
#   for df_chunk in reader:
#     print(df_chunk.dtypes)
#     break
#   # upload + export
#   upload_file_to_gcs(
#       drive_path,
#       f"ccd_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
#   )
#   export_to_gee(
#       f"ccd_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
#   )

.geo     object
cc_1      int64
cc_2      int64
cc_3      int64
cc_4      int64
cc_5      int64
cc_6      int64
cc_7      int64
cc_8      int64
cc_9      int64
cc_10     int64
cc_11     int64
cc_12     int64
cc        int64
dtype: object
Upload complete.
Ingestion task started: 2a4f86c7-03b8-4605-916d-fb63feae926e


# Combine CSVs CH

In [10]:
df = pd.read_csv('drive/MyDrive/TreeHealth/district_to_agroclimaticZone_mapping.csv')

In [11]:
# Function to convert string representation of list to an actual list
def convert_to_list(string):
    return ast.literal_eval(string)

df['IntersectingZones'] = df['IntersectingZones'].apply(convert_to_list)

In [12]:
district_mapping_df = df[df['AgroclimaticZone'] == agroclimatic_zone][['District', 'IntersectingZones']]

In [13]:
dist_list = []
i = 0
for ind in district_mapping_df.index:
    district = district_mapping_df.loc[ind, 'District']
    zones = district_mapping_df['IntersectingZones'][ind]
    # print(i, district, zones)
    dist_list.append(district)
    i += 1
# dist_list = ['Darjiling']
print(dist_list)

['Araria', 'Arwal', 'AurangabadB', 'Banka', 'Begusarai', 'Bhagalpur', 'Bhojpur', 'Buxar', 'Darbhanga', 'Gaya', 'Gopalganj', 'Jamui', 'Jehanabad', 'Kaimur', 'Katihar', 'Khagaria', 'Kishanganj', 'Lakhisarai', 'Madhepura', 'Madhubani', 'Munger', 'Muzaffarpur', 'Nalanda', 'Nawada', 'Pashchim Champaran', 'Patna', 'Purba Champaran', 'Purnia', 'Rohtas', 'Saharsa', 'Samastipur', 'Saran', 'Sheikhpura', 'Sheohar', 'Sitamarhi', 'Siwan', 'Supaul', 'Vaishali', 'Godda', 'Sahibganj', 'Ambedkar Nagar', 'Azamgarh', 'Bahraich', 'Ballia', 'Balrampur', 'Basti', 'Chandauli', 'Deoria', 'Faizabad', 'Ghazipur', 'Gonda', 'Gorakhpur', 'Jaunpur', 'Kushinagar', 'Maharajganj', 'Mau', 'Mirzapur', 'Sant Kabir Nagar', 'Sant Ravi Das Nagar', 'Shravasti', 'Siddharth Nagar', 'Sonbhadra', 'Varanasi']


In [14]:
def classify_chunk(df_chunk):
    """Add ch_class column based on rh50/rh75/rh98 logic."""
    conditions = [
        (df_chunk['rh50_class'] == 0) & (df_chunk['rh75_class'] == 0) & (df_chunk['rh98_class'] == 0),
        (df_chunk['rh50_class'] == 0) & (df_chunk['rh75_class'] == 0) & (df_chunk['rh98_class'] == 1),
        (df_chunk['rh50_class'] == 0) & (df_chunk['rh75_class'] == 1) & (df_chunk['rh98_class'] == 0),
        (df_chunk['rh50_class'] == 0) & (df_chunk['rh75_class'] == 1) & (df_chunk['rh98_class'] == 1),
        (df_chunk['rh50_class'] == 1) & (df_chunk['rh75_class'] == 0) & (df_chunk['rh98_class'] == 0),
        (df_chunk['rh50_class'] == 1) & (df_chunk['rh75_class'] == 0) & (df_chunk['rh98_class'] == 1)
    ]
    choices = [0, 0, 1, 2, 1, 2]

    df_chunk['ch_class'] = np.select(conditions, choices, default=3)
    return df_chunk

In [15]:
chunk_size=500_000
max_rows=80_000_000
for year in years:
    print(year)
    result_num = 0
    dst_num = 0
    row_counter = 0

    path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results CH/{year}/'
    os.makedirs(path, exist_ok=True)
    drive_path = f"{path}/result_{result_num}.csv"
    first_write = True  # controls header writing

    for district in dist_list:
        print(dst_num, district)
        dst_num += 1
        try:
            reader = pd.read_csv(
                f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_chm.csv',
                chunksize=chunk_size
            )
        except:
            print("Errorrrrrrrr")
            continue

        for df_chunk in reader:
            df_chunk = classify_chunk(df_chunk)

            # Append to current result file
            df_chunk.to_csv(drive_path, mode='a',
                            header=first_write, index=False)
            first_write = False

            row_counter += len(df_chunk)

            # If exceeds limit → finalize file and start new one
            if row_counter >= max_rows:
                print(f'Saving result_{result_num}.csv with {row_counter:,} rows')

                # upload + export
                upload_file_to_gcs(
                    drive_path,
                    f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
                )
                export_to_gee(
                    f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
                )

                # reset counters
                result_num += 1
                row_counter = 0
                drive_path = f"{path}/result_{result_num}.csv"
                first_write = True  # new file needs header

    # Flush leftover rows at end of loop
    if row_counter > 0:
        print(f'Saving result_{result_num}.csv with {row_counter:,} rows')
        upload_file_to_gcs(
            drive_path,
            f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
        )
        export_to_gee(
            f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
        )


2016
0 Araria
1 Arwal
2 AurangabadB
3 Banka
4 Begusarai
5 Bhagalpur
6 Bhojpur
7 Buxar
8 Darbhanga
9 Gaya
10 Gopalganj
11 Jamui
12 Jehanabad
13 Kaimur
14 Katihar
15 Khagaria
16 Kishanganj
17 Lakhisarai
18 Madhepura
19 Madhubani
20 Munger
21 Muzaffarpur
22 Nalanda
23 Nawada
24 Pashchim Champaran
25 Patna
26 Purba Champaran
27 Purnia
28 Rohtas
29 Saharsa
30 Samastipur
31 Saran
32 Sheikhpura
33 Sheohar
34 Sitamarhi
35 Siwan
36 Supaul
37 Vaishali
38 Godda
39 Sahibganj
40 Ambedkar Nagar
41 Azamgarh
42 Bahraich
43 Ballia
44 Balrampur
45 Basti
46 Chandauli
47 Deoria
48 Faizabad
49 Ghazipur
50 Gonda
51 Gorakhpur
52 Jaunpur
53 Kushinagar
54 Maharajganj
55 Mau
56 Mirzapur
57 Sant Kabir Nagar
58 Sant Ravi Das Nagar
59 Shravasti
60 Siddharth Nagar
61 Sonbhadra
62 Varanasi
Saving result_0.csv with 68,660,369 rows


KeyboardInterrupt: 

In [ ]:
# for year in years:
#   print(year)
#   df_con = pd.DataFrame()
#   df_len = 0
#   result_num = 0
#   dst_num = 0
#   for district in dist_list:
#     print(dst_num, district)
#     dst_num += 1
#     try:
#         df = pd.read_csv(f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_chm.csv')
#     except:
#         continue
#     df_len += len(df)
#     df_con = pd.concat([df_con, df])
#     del(df)
#     print(df_len)

#     if df_len > 80000000:
#         print(f'Saving result_{result_num}.csv')

#         conditions = [
#             (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 0),
#             (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 1),
#             (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 1) & (df_con['rh98_class'] == 0),
#             (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 1) & (df_con['rh98_class'] == 1),
#             (df_con['rh50_class'] == 1) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 0),
#             (df_con['rh50_class'] == 1) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 1)
#         ]

#         choices = [0, 0, 1, 2, 1, 2]

#         df_con['ch_class'] = np.select(conditions, choices, default=3)

#         path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results CH/{year}/'
#         if not os.path.exists(path):
#             os.makedirs(path)
#         drive_path = f'{path}/result_{result_num}.csv'
#         df_con.to_csv(drive_path, index=False)

#         upload_file_to_gcs(drive_path, f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}")
#         export_to_gee(f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}")

#         result_num += 1
#         del(df_con)
#         df_len = 0
#         df_con = pd.DataFrame()

#   if (len(df_con) > 0):
#     print(f'Saving result_{result_num}.csv')

#     conditions = [
#         (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 0),
#         (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 1),
#         (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 1) & (df_con['rh98_class'] == 0),
#         (df_con['rh50_class'] == 0) & (df_con['rh75_class'] == 1) & (df_con['rh98_class'] == 1),
#         (df_con['rh50_class'] == 1) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 0),
#         (df_con['rh50_class'] == 1) & (df_con['rh75_class'] == 0) & (df_con['rh98_class'] == 1)
#     ]

#     choices = [0, 0, 1, 2, 1, 2]

#     df_con['ch_class'] = np.select(conditions, choices, default=3)

#     path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results CH/{year}/'
#     if not os.path.exists(path):
#         os.makedirs(path)

#     drive_path = f'{path}/result_{result_num}.csv'
#     df_con.to_csv(drive_path, index=False)

#     upload_file_to_gcs(drive_path, f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}")
#     export_to_gee(f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}")

#     result_num += 1
#     del(df_con)
#     df_len = 0
#     df_con = pd.DataFrame()

In [ ]:

# year = '2023'
# path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/Compiled Results CH/{year}/'
# for result_num in range(6):
#   drive_path = f"{path}/result_{result_num}.csv"
#   # upload + export
#   upload_file_to_gcs(
#       drive_path,
#       f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
#   )
#   export_to_gee(
#       f"ch_{year}_result_{result_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
#   )